# PopQuants — Strategy 3 | Real SEC 13F + Word2Vec + Triangulated Statistical Arbitrage
## เรียนทีละขั้นจากความคิดง่าย ๆ → สมการ → ผลคำนวณจากข้อมูลจริง
**ตัวอย่างผล Backtest ปี 2026:** 2 ม.ค.–19 พ.ย. 2026, 223 วัน, Total Return **+21.20%**, Sharpe (rf=0) **1.611**, Max Drawdown **−4.78%**, Annual Vol **14.11%**.  
**ยังไม่ผ่าน Research Validation:** Gensim `our_dot_float` warnings, Universe เพียง 4 หุ้น, Pair Depth ต่ำ, 13F Top-160 truncation และยังไม่มี Stock Borrow Costs.

**กติกาการสอน:** 1) ตัวเลขของคำ/เวกเตอร์ด้านล่างเป็น *ตัวอย่างอธิบายคณิตศาสตร์* 2) ตารางที่อ่านจาก `~/Documents/usmart/...` เป็น *ข้อมูล SEC และ Backtest จริงบนเครื่องสำหรับประมวลผล* 3) อย่าสับสนว่า 3 มิติที่ PCA แสดงเป็นเวกเตอร์ Word2Vec ดั้งเดิม (โมเดลจริงมี 48 มิติ).

## ส่วน A — Word2Vec และการเรียนรู้ความสัมพันธ์จากบริบท
เราไม่ได้ต้องการให้ AI เดาราคาพรุ่งนี้ แต่ต้องการให้ AI ช่วยตอบว่า **หุ้นคู่ใดมีบริบทการถือครองของสถาบันใกล้กัน** ก่อนนำไปทดสอบ Spread Mean Reversion.

ลองคิดถึงคำว่า **แมว / สุนัข / รถยนต์** ถ้า AI อ่านประโยคจำนวนมากและเห็น "แมว" กับ "สุนัข" ในบริบทคล้ายกัน มันอาจเรียนรู้เวกเตอร์ที่ชี้ไปใกล้กัน ส่วน "รถยนต์" อาจอยู่ไกลออกไป *ในมิติทางความหมาย*.

In [ ]:
# ตัวอย่างที่สร้างขึ้นเพื่ออธิบายเท่านั้น ไม่ใช่ Word2Vec ที่ Train จริง
import numpy as np, pandas as pd, plotly.graph_objects as go
word_vectors = pd.DataFrame({
    'คำ':['แมว','สุนัข','รถยนต์'],
    'มิติ_สัตว์':[0.90,0.85,0.10],
    'มิติ_เลี้ยงในบ้าน':[0.80,0.75,0.05],
    'มิติ_พาหนะ':[0.05,0.10,0.98]
}).set_index('คำ')
print('ตาราง 1: เวกเตอร์คำสมมติใน 3 มิติ'); display(word_vectors)

### Cosine Similarity — วัด "มุม" ระหว่างเวกเตอร์
\[
\mathrm{cosine}(A,B)=\frac{A\cdot B}{\|A\|\,\|B\|}
\]
- **A, B** = เวกเตอร์ของสิ่งสองอย่างที่เราจะเทียบ
- **A·B** = คูณสมาชิกตำแหน่งเดียวกัน แล้วนำมาบวก
- **||A||** = ความยาวเวกเตอร์ A = รากที่สองของผลรวมกำลังสอง
- **ค่าคอสไซน์** ใกล้ +1 หมายถึงทิศทางใกล้กัน, 0 หมายถึงตั้งฉาก, −1 หมายถึงทิศตรงข้าม

**ตัวอย่าง:** A=(1,0,0), B=(0.8,0.6,0): A·B=0.8, ||A||=1, ||B||=1 ⇒ cosine=0.8

In [ ]:
from numpy.linalg import norm
A=np.array([1.0,0.0,0.0]);B=np.array([0.8,0.6,0.0])
dot=(A*B).sum(); length_a=norm(A);length_b=norm(B)
print(f'A·B = {dot:.3f}; |A| = {length_a:.3f}; |B| = {length_b:.3f}')
print(f'cos(A,B) = {dot:.3f} / ({length_a:.3f} × {length_b:.3f}) = {dot/(length_a*length_b):.3f}')
print('ตาราง 2: Cosine ของคำตัวอย่าง')
V=word_vectors.to_numpy();unit=V/norm(V,axis=1)[:,None]
display(pd.DataFrame(unit@unit.T,index=word_vectors.index,columns=word_vectors.index).round(3))

In [ ]:
# ตัวอย่างภาพ 3 มิติจากตัวเลขสมมติ เพื่อให้เห็นมุม/ทิศทาง
f=go.Figure()
for label,row in word_vectors.iterrows():
    xyz=row.to_numpy(float)
    f.add_trace(go.Scatter3d(x=[0,xyz[0]],y=[0,xyz[1]],z=[0,xyz[2]],mode='lines+markers+text',
       text=['',label],textposition='top center',name=label))
f.update_layout(title='ภาพเวกเตอร์ 3 มิติ (ตัวอย่างคำสมมติ ไม่ใช่ข้อมูล SEC)',
    scene=dict(xaxis_title='สัตว์',yaxis_title='สัตว์เลี้ยง',zaxis_title='พาหนะ'),height=520)
f.show()

## ส่วน B — จาก Word2Vec ไปสู่ SEC 13F จริง
Form 13F คือรายงานหุ้นที่ผู้จัดการลงทุนสถาบันถืออยู่ในช่วงรายงาน (ไม่ใช่ข้อมูลธุรกรรมรายวัน)

**Fund as Sentence:** กองทุนหนึ่ง = หนึ่งประโยค, หุ้นในพอร์ต = คำหนึ่งคำ; Word2Vec เรียนรู้ว่าสินทรัพย์ใดปรากฏร่วมในบริบทคล้ายกัน

**ข้อจำกัดสำคัญ:** ต้องกำหนดวันที่ข้อมูลเปิดเผยจริง (Point-in-Time) ก่อนอนุญาตให้โมเดลใช้ข้อมูลนั้น และการเลือกเฉพาะ Top-160 ต่อ Filing เป็น approximation.

In [ ]:
from pathlib import Path
ROOT=Path.home()/'Documents'/'usmart'
OUT=ROOT/'popquants_strategy3_real13f_2026'
required=['daily_backtest.csv','selected_pairs.csv','quarter_model_audit.csv','aggregate_signals.csv',
          'depth.csv','consistency.csv','target_weights.csv','metrics.json']
print('ตรวจสอบไฟล์จากการรัน Strategy 3 จริง:')
for filename in required: print(('OK ' if (OUT/filename).exists() else 'MISSING '), OUT/filename)
assert all((OUT/f).exists() for f in required), 'กรุณารัน popquants_strategy3_2026_teaching.py ก่อน (ใช้ SEC จริง)'

In [ ]:
# ตาราง 3: ตรวจว่ากองทุนแต่ละไตรมาสมีกี่กองทุน และมีคู่หุ้นที่ผ่านกี่คู่
audit=pd.read_csv(OUT/'quarter_model_audit.csv'); display(audit)
print('หมายเหตุ: ฝึก Word2Vec จาก Holdings ของ SEC ที่ผ่านเงื่อนไข Filing Date; ไม่ใช่ข้อมูลจำลอง')

In [ ]:
# ตาราง 4: คู่หุ้นที่ได้รับเลือกจริงจาก Word2Vec (แต่ละช่วงเวลา)
pairs=pd.read_csv(OUT/'selected_pairs.csv');display(pairs.head(30))
print('จำนวนคู่ในแต่ละช่วง:');display(pairs.groupby('period').agg(pair_count=('pair','nunique')))

## ส่วน C — Word2Vec และเวกเตอร์จริง
Word2Vec ใน Script ใช้ **48 มิติ** ไม่ใช่ 3 มิติ; ภาพ 3 มิติต่อไปเกิดจาก **PCA projection** ของเวกเตอร์ที่เรียนรู้มาแล้ว โดยเหมาะกับการสื่อสาร *ไม่ใช่ใช้แทน Cosine Similarity ใน 48 มิติ*.

เช่นหุ้น AMAT กับ LRCX อยู่ใกล้กันในภาพ 3 มิติ ไม่ได้ยืนยันว่า Cosine ของต้นฉบับสูงเสมอ เพราะการลดมิติอาจบิดรูปทรง.

In [ ]:
pca_file=OUT/'embedding_3d_pca.csv'
if pca_file.exists():
    emb=pd.read_csv(pca_file)
    if len(emb):
        last=sorted(emb.period.unique())[-1]
        sample=emb[emb.period.eq(last)].copy()
        print('ตาราง 5: PCA projection จาก Word2Vec ฝึกด้วย SEC จริง ณ',last)
        display(sample[['ticker','cusip','pc1','pc2','pc3','is_target']].head(20))
        ff=go.Figure(go.Scatter3d(x=sample.pc1,y=sample.pc2,z=sample.pc3,
          mode='markers+text',text=sample.ticker.fillna(''),
          marker=dict(size=np.where(sample.is_target,9,3),color=np.where(sample.is_target,'#E8BE62','#61CFB6')),
          hovertext=sample.cusip,hovertemplate='%{hovertext}<extra></extra>'))
        ff.update_layout(title='Real SEC 13F Word2Vec: 3D PCA projection (เพื่ออธิบาย)',height=550)
        ff.show()
    else: print('PCA file is empty; see numerical warnings')
else: print('ยังไม่พบ embedding_3d_pca.csv; ให้รันสคริปต์ Teaching Version ใหม่')

## ส่วน D — ราคาหุ้น → Spread → Z-score
กลยุทธ์ไม่เอาราคา AMAT (เช่น $100) ลบราคา LRCX ($150) ตรง ๆ เพื่อสรุปถูกแพง เพราะราคาหุ้นต่างขนาดกัน

เราเริ่มจาก **log-adjusted prices**:
\[
S_t=\ln(P_{A,t})-\ln(P_{B,t})
\]
- **S_t** = Spread ณ วัน t; **P_A,P_B** = ราคาปรับ Corporate Actions ของหุ้น A,B
- **ln** = Natural Log; ผลต่าง Log เท่ากับ Log ของอัตราส่วนราคา

จากนั้นเทียบ Spread ปัจจุบันกับค่าเฉลี่ยในอดีต:
\[
Z_t=\frac{S_t-\mu_{t-1}}{\sigma_{t-1}}
\]
- **μ** = ค่าเฉลี่ย Spread ในหน้าต่าง 60 วันก่อนหน้า
- **σ** = Standard Deviation ของ Spread ในหน้าต่างเดียวกัน
- ใช้ข้อมูลอดีตด้วย `shift(1)` เพื่อไม่ให้รวมค่า Spread ปัจจุบันในการสร้าง baseline

In [ ]:
prices=pd.read_csv(ROOT/'popquants_strategy1'/'us_stock_adjusted_daily.csv')
prices['date']=pd.to_datetime(prices['date']);prices=prices.set_index('date').sort_index()
print('ตาราง 6: ราคาปรับย้อนหลังจาก uSMART (ตัวอย่าง 5 วันแรก)');display(prices[['AMAT','LRCX','KLAC','TER']].head())
s=np.log(prices['AMAT'])-np.log(prices['LRCX'])
mu=s.rolling(60,min_periods=60).mean().shift(1)
sd=s.rolling(60,min_periods=60).std().shift(1)
z=(s-mu)/sd.replace(0,np.nan)
spread_table=pd.DataFrame({'ราคา_AMAT':prices.AMAT,'ราคา_LRCX':prices.LRCX,'log_spread':s,'mean_อดีต':mu,'sd_อดีต':sd,'Z':z})
valid_spread=spread_table.dropna();display(valid_spread.tail(8).round(4))
last=valid_spread.iloc[-1]
print(f"ตัวอย่างแทนค่า Z วันล่าสุด: ({last['log_spread']:.4f} - {last['mean_อดีต']:.4f}) / {last['sd_อดีต']:.4f} = {last['Z']:.3f}")

## ส่วน E — หนึ่ง Spread ออกเสียงสองฝั่ง (Votes)
หาก AMAT/LRCX มี Z = +2.0 หมายถึง **AMAT ดู Rich เทียบ LRCX**:
- AMAT ได้ **+2.0** (Short candidate)
- LRCX ได้ **−2.0** (Long candidate)

พอมีหลายคู่ เราได้คะแนนหุ้นรายตัว:
\[
Signal_i=\mathrm{mean}(Votes_i),\quad Depth_i=\text{จำนวนคู่ที่ให้ Vote},\quad Consistency_i=\left|\mathrm{mean}(sign(Votes_i))\right|
\]
**เช่น:** หุ้น A ได้ Votes [+2,+1,−1] ⇒ Signal = (2+1−1)/3 = +0.667, Depth=3, Consistency=|(1+1−1)/3|=0.333.  
**สำคัญ:** เวอร์ชันสอนนี้ใช้คะแนนเฉลี่ยแบบน้ำหนักเท่ากัน; ไม่ใช่ Sharpe-weighted Votes ตามบทความฉบับเต็ม.

In [ ]:
agg=pd.read_csv(OUT/'aggregate_signals.csv',parse_dates=['date']).set_index('date')
dep=pd.read_csv(OUT/'depth.csv',parse_dates=['date']).set_index('date')
con=pd.read_csv(OUT/'consistency.csv',parse_dates=['date']).set_index('date')
tgt=pd.read_csv(OUT/'target_weights.csv',parse_dates=['date']).set_index('date')
# เลือกวันแรกที่มีการถือสถานะ เพื่อเห็นว่าระบบตัดสินใจจริงอย่างไร
active=tgt.abs().sum(axis=1)>0
assert active.any(),'ยังไม่พบวันที่เปิดสถานะในผลทดสอบ'
day=active[active].index[0]
print('ตาราง 7: คะแนนจาก SEC Word2Vec + ราคาหุ้น ณ วันที่โมเดลตัดสินใจ',day.date())
view=pd.DataFrame({'Aggregate':agg.loc[day], 'Depth':dep.loc[day], 'Consistency':con.loc[day], 'TargetWeight':tgt.loc[day]})
display(view.round(3))

## ส่วน F — Portfolio และต้นทุน
โมเดลเรียง Aggregate ที่ผ่าน Depth/Consistency แล้วเลือกหนึ่งตัว Long (+0.5) และหนึ่งตัว Short (−0.5)

**Gross Daily Return:**
\[ R_{gross,t}=\sum_i w_{i,t-1}r_{i,t}\]
- **w_{i,t-1}** = น้ำหนักที่ตัดสินใจจากข้อมูลวันก่อนหน้า
- **r_{i,t}** = ผลตอบแทนของหุ้น i ณ วัน t

**Turnover:** \(T_t=\sum_i|w_{i,t}-w_{i,t-1}|\)

**Net Return:** \(R_{net,t}=R_{gross,t}-T_t\times 0.0005\) โดย 0.0005 = 5 bps ต่อ 1 หน่วย Turnover

**ตัวอย่างทั่วไป:** Gross +1.00%, Turnover 1 ⇒ Cost 0.05% ⇒ Net +0.95%.  
**คำเตือน:** ยังไม่ได้หัก Borrow Fee, Financing และ Dividend on shorts.

In [ ]:
daily=pd.read_csv(OUT/'daily_backtest.csv',parse_dates=['date']).set_index('date')
print('ตาราง 8: Gross, Turnover และ Net (ล่าสุด 10 วัน)')
display(daily[['gross_return','turnover','net_return']].tail(10).round(5))
check=(daily.gross_return-daily.turnover*0.0005-daily.net_return)
print('ตรวจ Gross − Cost − Net (เว้นวันสุดท้ายที่สคริปต์เดิมไม่บันทึกต้นทุนจากสัญญาณที่ยังไม่ execute):')
print('max abs residual excluding last:',check.iloc[:-1].abs().max())

## ส่วน G — Performance: ผลตอบแทนและความเสี่ยง
\[
Equity_t=\prod_{k=1}^{t}(1+R_{net,k})
\]
\[
Sharpe=\frac{mean(R_{net})}{std(R_{net})}\sqrt{252}\qquad (rf=0)
\]
\[
Drawdown_t=\frac{Equity_t}{\max_{s\le t}Equity_s}-1
\]
- **mean** = ผลตอบแทนเฉลี่ยรายวัน
- **std** = Standard Deviation ของผลตอบแทนรายวัน
- **252** = จำนวนวันซื้อขายประมาณหนึ่งปีที่ใช้ Annualize
- **Drawdown** = พอร์ตลดจากจุดสูงสุดก่อนหน้านั้นเท่าไร
- **Max Drawdown** = ค่าที่ติดลบมากที่สุดของ Drawdown

In [ ]:
import json
metrics=json.loads((OUT/'metrics.json').read_text())
r=daily.net_return.astype(float);equity=(1+r).cumprod(); draw=equity/equity.cummax()-1
sharpe=r.mean()/r.std(ddof=1)*np.sqrt(252)
summary=pd.DataFrame({'Metric':['Total Return %','Annual Volatility %','Sharpe rf=0','Max Drawdown %','Trading days'],
  'Recomputed':[100*(equity.iloc[-1]-1),100*r.std(ddof=1)*np.sqrt(252),sharpe,100*draw.min(),len(r)],
  'Saved run':[metrics['total_return_pct'],metrics['annualized_vol_pct'],metrics['sharpe_rf0'],metrics['max_drawdown_pct'],metrics['days']]})
print('ตาราง 9: คำนวณ Metrics ซ้ำจาก CSV จริง เทียบ metrics.json');display(summary.round(4))

In [ ]:
import plotly.graph_objects as go
fig=go.Figure();fig.add_trace(go.Scatter(x=equity.index,y=equity,name='Strategy 3 NET'))
fig.update_layout(title='Equity — SEC 13F + Word2Vec 2026 (local backtest)',height=360)
fig.show()
fig2=go.Figure(go.Scatter(x=draw.index,y=100*draw,fill='tozeroy'))
fig2.update_layout(title='Drawdown (%)',height=330);fig2.show()

## ส่วน H — ความแตกต่างระหว่าง Strategy 2 และ Strategy 3
**เหมือนกัน:** Spread → Z-score → Votes → Long/Short Portfolio

**ต่างกัน:** Strategy 2 เริ่มจาก Industry/Economic Relationships แบบกำหนดไว้ ส่วน Strategy 3 ใช้ Word2Vec ที่ฝึกจาก SEC 13F เพื่อเลือก Pair ในแต่ละไตรมาส

จุดประสงค์ของการทดลองคือ *เปลี่ยนวิธีเลือก Pair* แล้วดูว่า Risk-adjusted Return ต่างกันเพราะอะไร แต่ต้องควบคุม Universe, ช่วงทดสอบ, Costs และ Signal Rules ให้ใกล้เคียงกันจริง ๆ

In [ ]:
for f in ['performance_comparison.csv','quarter_model_audit.csv']:
    p=OUT/f
    if p.exists():
        print('ตารางเปรียบเทียบ / Audit:',f)
        display(pd.read_csv(p).head(20))
print('ข้อสังเกต: ถ้าช่วงทดสอบหรือ Exposure ต่างกัน การเปรียบเทียบยังไม่ใช่ Controlled Experiment')

## ส่วน I — Scientific checks ที่ยังต้องผ่าน
1. **Numerical stability:** Gensim `our_dot_float` exceptions ต้องได้รับการวินิจฉัยและเทียบผลกับเวอร์ชัน/การคำนวณอื่นก่อน
2. **Universe:** 4 หุ้นเล็กเกินไป, Depth=1 บางช่วงไม่ได้ triangulate จริง
3. **Point-in-time:** ตรวจวันที่ Filing จริงและการนำข้อมูลไปใช้หลังเผยแพร่; 45 วันเป็นสมมติฐานอนุรักษนิยม
4. **Holdings:** Top-160 ไม่ใช่พอร์ตเต็ม; 13F ไม่ครอบคลุม Short และทุกสินทรัพย์
5. **Execution:** ราคา ณ Close → การซื้อขายวันถัดไปเป็น Proxy; ต้องทดสอบ Open/Slippage/Borrow
6. **Selection & OOS:** ยังไม่ใช่ผล Holdout อิสระ, ไม่ใช่การยืนยัน Live Edge

**สิ่งที่ผู้เรียนได้:** สร้างเส้นทางตั้งแต่ข้อมูลดิบ → เวกเตอร์หุ้น → คู่หุ้น → สัญญาณ → PnL และรู้วิธีตรวจสอบความน่าเชื่อถือของงานวิจัย

In [ ]:
# เปิด HTML ที่สร้างจากข้อมูล SEC จริงโดยไม่ส่งคำสั่งซื้อขาย
from IPython.display import HTML, display
from pathlib import Path
p=OUT/'interactive_tearsheet_detailed.html'
print('Interactive report:',p,'exists:',p.exists())
print('เปิดได้โดยคำสั่ง Terminal: open',repr(str(p)))